# TP2_1 — DCGAN on Lego Brick Images
Train a Deep Convolutional GAN to generate **64×64 grayscale** images of Lego bricks.

## 0. Install & Download Dataset

In [ ]:
# Run once to install kaggle and download the dataset
# !pip install kaggle
# !kaggle datasets download -d joosthazelzet/lego-brick-images --unzip -p ./lego_dataset

## 1. Imports & Hyperparameters

In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
import torchvision.utils as vutils
from torch.utils.data import DataLoader, Dataset, Subset
from PIL import Image
from tqdm import tqdm
import numpy as np
import matplotlib.pyplot as plt

# --------------------
# Hyperparameters
# --------------------
IMAGE_SIZE    = 64
CHANNELS      = 1          # Grayscale
BATCH_SIZE    = 64         # Réduit pour aller plus vite
Z_DIM         = 100
EPOCHS        = 20         # Réduit (100 → 20) pour CPU
LEARNING_RATE = 0.0002
BETA1         = 0.5
BETA2         = 0.999
NOISE_PARAM   = 0.1
MAX_IMAGES    = 5000       # On prend 5000 images au lieu de 40000
DEVICE        = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

## 2. Transforms

**Why normalize to [−1, 1] ?**  
The Generator's final activation is `Tanh`, which outputs values in [−1, 1].  
To be consistent, we must normalize real images to the same range so that the Discriminator
sees inputs on the same scale from both paths.  
Using `Normalize(mean=[0.5], std=[0.5])` maps `[0, 1] → [−1, 1]`.

In [ ]:
transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),   # 1 channel
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),    # 64x64
    transforms.ToTensor(),                         # [0,1]
    transforms.Normalize(mean=[0.5], std=[0.5])    # [-1,1]
])

## 3. Dataset & DataLoader
Custom dataset that returns images (no labels) from a flat folder.

In [ ]:
from torch.utils.data import DataLoader, Dataset, Subset

# --------------------
# Dataset & DataLoader
# --------------------
class ImageDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir  = root_dir
        self.transform = transform
        self.image_files = sorted([
            f for f in os.listdir(root_dir)
            if f.endswith(('.png', '.jpg', '.jpeg'))
        ])

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        img_path = os.path.join(self.root_dir, self.image_files[idx])
        image = Image.open(img_path).convert("RGB")
        if self.transform:
            image = self.transform(image)
        return image

# Images are in lego_dataset/dataset/
dataset_path = "./lego_dataset/dataset"

full_dataset = ImageDataset(root_dir=dataset_path, transform=transform)

# Sous-ensemble de MAX_IMAGES pour accélérer l'entraînement sur CPU
indices = torch.randperm(len(full_dataset))[:MAX_IMAGES].tolist()
dataset = Subset(full_dataset, indices)

dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True,
                        drop_last=True, num_workers=0)
print(f"Dataset size: {len(dataset)} images — {len(dataloader)} batches/epoch")

## 4. Plot Sample Images

In [ ]:
def plot_sample_images(loader, n=16):
    batch = next(iter(loader))          # (B, 1, 64, 64)
    batch = batch[:n].cpu()
    grid  = vutils.make_grid(batch, nrow=8, normalize=True)
    plt.figure(figsize=(12, 4))
    plt.axis("off")
    plt.title("Real Lego Images (sample)")
    plt.imshow(grid.permute(1, 2, 0).squeeze(), cmap="gray")
    plt.tight_layout()
    plt.show()

plot_sample_images(dataloader)

## 5. GAN Architecture

### 5.1 Discriminator
- Input: 64×64 grayscale image  
- 4 strided Conv2d layers (×2 downsampling each) → 1×1  
- BatchNorm after layer 2, 3, 4  
- LeakyReLU(0.2) + Dropout(0.3) after each layer  
- Sigmoid output → probability in [0, 1]

In [ ]:
# ------------------------------
# Discriminator
# ------------------------------
class Discriminator(nn.Module):
    def __init__(self):
        super(Discriminator, self).__init__()
        self.model = nn.Sequential(
            # (CHANNELS, 64, 64) -> (64, 32, 32)
            nn.Conv2d(CHANNELS, 64, 4, 2, 1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Dropout(0.3),

            # (64, 32, 32) -> (128, 16, 16)
            nn.Conv2d(64, 128, 4, 2, 1, bias=False),
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Dropout(0.3),

            # (128, 16, 16) -> (256, 8, 8)
            nn.Conv2d(128, 256, 4, 2, 1, bias=False),
            nn.BatchNorm2d(256),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Dropout(0.3),

            # (256, 8, 8) -> (512, 4, 4)
            nn.Conv2d(256, 512, 4, 2, 1, bias=False),
            nn.BatchNorm2d(512),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Dropout(0.3),

            # (512, 4, 4) -> (1, 1, 1)
            nn.Conv2d(512, 1, 4, 1, 0, bias=False),
            nn.Sigmoid(),
            nn.Flatten(start_dim=1)   # (batch_size, 1)
        )

    def forward(self, x):
        return self.model(x)

### 5.2 Generator
- Input: noise vector z of shape (Z_DIM, 1, 1)  
- 5 transposed Conv layers (×2 upsampling each) → 64×64  
- BatchNorm2d + LeakyReLU(0.2) after each layer except the last  
- Tanh final activation → output in [−1, 1]

In [ ]:
# ------------------------------
# Generator
# ------------------------------
class Generator(nn.Module):
    def __init__(self):
        super(Generator, self).__init__()
        self.model = nn.Sequential(
            # (Z_DIM, 1, 1) -> (512, 4, 4)
            nn.ConvTranspose2d(Z_DIM, 512, 4, 1, 0, bias=False),
            nn.BatchNorm2d(512),
            nn.LeakyReLU(0.2, inplace=True),

            # (512, 4, 4) -> (256, 8, 8)
            nn.ConvTranspose2d(512, 256, 4, 2, 1, bias=False),
            nn.BatchNorm2d(256),
            nn.LeakyReLU(0.2, inplace=True),

            # (256, 8, 8) -> (128, 16, 16)
            nn.ConvTranspose2d(256, 128, 4, 2, 1, bias=False),
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.2, inplace=True),

            # (128, 16, 16) -> (64, 32, 32)
            nn.ConvTranspose2d(128, 64, 4, 2, 1, bias=False),
            nn.BatchNorm2d(64),
            nn.LeakyReLU(0.2, inplace=True),

            # (64, 32, 32) -> (CHANNELS, 64, 64)
            nn.ConvTranspose2d(64, CHANNELS, 4, 2, 1, bias=False),
            nn.Tanh()   # output in [-1, 1]
        )

    def forward(self, x):
        return self.model(x)

# Instantiate models
critic    = Discriminator().to(DEVICE)
generator = Generator().to(DEVICE)

print("Discriminator:")
print(critic)
print("\nGenerator:")
print(generator)

## 6. Losses & Optimizers

In [ ]:
# ------------------------
# Losses & Optimizers
# ------------------------
criterion   = nn.BCELoss()   # Binary Cross-Entropy

d_optimizer = optim.Adam(critic.parameters(),    lr=LEARNING_RATE, betas=(BETA1, BETA2))
g_optimizer = optim.Adam(generator.parameters(), lr=LEARNING_RATE, betas=(BETA1, BETA2))

## 7. Training Loop

In [ ]:
# ------------------------
# Training
# ------------------------
g_loss_values = []
d_loss_values = []
g_acc_values  = []
d_acc_values  = []

for epoch in range(EPOCHS):
    g_loss_batch = []
    d_loss_batch = []
    g_acc_batch  = []
    d_acc_batch  = []

    progress_bar = tqdm(enumerate(dataloader), total=len(dataloader),
                        desc=f"Epoch {epoch+1}/{EPOCHS}")

    for i, real_images in progress_bar:
        real_images = real_images.to(DEVICE)
        batch_size  = real_images.size(0)

        # -------------------------
        # 1) Train Discriminator
        # -------------------------
        d_optimizer.zero_grad()

        # True labels near 1, with some noise
        real_labels       = torch.ones(batch_size, 1, device=DEVICE)
        real_noisy_labels = real_labels - NOISE_PARAM * torch.rand_like(real_labels)

        # Fake labels near 0, with some noise
        fake_labels       = torch.zeros(batch_size, 1, device=DEVICE)
        fake_noisy_labels = fake_labels + NOISE_PARAM * torch.rand_like(fake_labels)

        # Real images
        real_preds = critic(real_images)
        real_loss  = criterion(real_preds, real_noisy_labels)

        # Fake images
        z           = torch.randn(batch_size, Z_DIM, 1, 1, device=DEVICE)
        fake_images = generator(z)
        fake_preds  = critic(fake_images.detach())
        fake_loss   = criterion(fake_preds, fake_noisy_labels)

        d_loss = (real_loss + fake_loss) / 2
        d_loss.backward()
        d_optimizer.step()

        # Discriminator accuracy
        d_real_acc = (real_preds.round() == real_labels).float().mean().item()
        d_fake_acc = (fake_preds.round() == fake_labels).float().mean().item()
        d_acc      = (d_real_acc + d_fake_acc) / 2

        # -------------------------
        # 2) Train Generator
        # -------------------------
        g_optimizer.zero_grad()
        fake_preds = critic(fake_images)          # no detach: gradients flow to G
        # Generator wants D to say "real" => target = 1
        g_loss = criterion(fake_preds, real_labels)
        g_loss.backward()
        g_optimizer.step()

        g_acc = (fake_preds.round() == real_labels).float().mean().item()

        # Track batch metrics
        d_loss_batch.append(d_loss.item())
        g_loss_batch.append(g_loss.item())
        d_acc_batch.append(d_acc)
        g_acc_batch.append(g_acc)

        progress_bar.set_postfix(D_Loss=f"{d_loss.item():.4f}",
                                 G_Loss=f"{g_loss.item():.4f}")

    # End of epoch: record averages
    d_loss_values.append(np.mean(d_loss_batch))
    g_loss_values.append(np.mean(g_loss_batch))
    d_acc_values.append(np.mean(d_acc_batch))
    g_acc_values.append(np.mean(g_acc_batch))

    # Save sample images every 10 epochs
    if (epoch + 1) % 10 == 0:
        with torch.no_grad():
            z_sample    = torch.randn(16, Z_DIM, 1, 1, device=DEVICE)
            fake_sample = generator(z_sample).cpu()
        vutils.save_image(fake_sample, f"fake_sample_epoch_{epoch+1:03d}.png",
                          normalize=True)
        print(f"  → saved fake_sample_epoch_{epoch+1:03d}.png")

### (Optional) Load a pre-trained model instead of training

In [ ]:
# GEN_PATH   = "generator.pth"
# CRITIC_PATH = "discriminator.pth"
# generator.load_state_dict(torch.load(GEN_PATH, map_location=DEVICE))
# critic.load_state_dict(torch.load(CRITIC_PATH, map_location=DEVICE))
# generator.eval()
# critic.eval()
# print("Pre-trained models loaded.")

## 8. Monitoring Progress — Loss & Accuracy Curves

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes = axes.flatten()

titles = ['epoch_g_loss', 'epoch_g_acc', 'epoch_d_loss', 'epoch_d_acc']
data   = [g_loss_values, g_acc_values, d_loss_values, d_acc_values]

for ax, title, values in zip(axes, titles, data):
    ax.plot(values, color='brown')
    ax.set_title(title)
    ax.set_xlabel("Epoch")
    ax.grid(True)

plt.tight_layout()
plt.show()

### Analysis Questions

**What do you observe? What's wrong?**  
A well-trained GAN should show both losses converging toward equilibrium (~0.69 for BCE with balanced predictions). If the discriminator loss collapses to near 0 quickly while the generator loss stays high or oscillates, the discriminator has overpowered the generator — it easily distinguishes real from fake, so the gradient signal to the generator becomes uninformative.

**If the discriminator loss is near zero but the generator loss is large, what does that imply?**  
The discriminator perfectly classifies every fake image as fake. It has learned the distribution much faster than the generator can learn to fool it. The generator receives near-zero gradients (vanishing gradient problem), making training stall. The label noise (`NOISE_PARAM`) and Dropout in the discriminator are tricks to slow it down and prevent this.

**What does "mode collapse" look like?**  
The generator produces the same image (or a very small set of images) regardless of the input noise vector z. In the generated grid, all images look nearly identical. In the loss curves, the generator loss may appear stable but low, while the discriminator loss rises again — the discriminator eventually notices all fakes are identical. Mode collapse is one of the central failure modes of standard GANs.

## 9. Generating & Evaluating

In [ ]:
# --------------------------
# Sampling — 8×4 grid
# --------------------------
grid_width, grid_height = 8, 4

generator.eval()
with torch.no_grad():
    z_sample = torch.randn(grid_width * grid_height, Z_DIM, 1, 1, device=DEVICE)
    gen_imgs  = generator(z_sample).cpu().detach()

fig = plt.figure(figsize=(grid_width * 1.5, grid_height * 1.5))
for i in range(grid_width * grid_height):
    ax = fig.add_subplot(grid_height, grid_width, i + 1)
    ax.axis("off")
    # gen_imgs shape: (N, 1, 64, 64) — channel 0 is grayscale
    ax.imshow(gen_imgs[i, 0, :, :], cmap="gray", vmin=-1, vmax=1)

plt.suptitle("Generated Lego Bricks", fontsize=14)
plt.tight_layout()
plt.show()

## 10. Save Models

In [ ]:
torch.save(generator.state_dict(), "generator.pth")
torch.save(critic.state_dict(),    "discriminator.pth")
print("Models saved.")